#### List Operations
- Indexing: Modify (rebind) a single item in a list by assigning to an indexing: `xs[1] = 42` makes the element at index `1` `42`. This assumes that you have allocated enough memory first, otherwise you will get an `IndexError`:

In [1]:
xs = []
xs[0] = 1

IndexError: list assignment index out of range

In [2]:
xs = [1, 2, 3]
xs[1] = 20 # A list is an indexed data structure, make sure you have room for the data before inserting it
xs

[1, 20, 3]

- Slicing: _Using slices as targets_  is tricky. Remember to provide slices on the RHS when if you specify a slice as the target on the LHS. This feature is strange and makes the code more cryptic. Readers could easily miss a `:` inside the subscript. It may also _delete_ list members!

In [3]:
xs = [1, 3, 5, 7]
xs[:] = [2, 4, 6, 8] # unbinds xs from the object it was bound to before and rebinds it to the expression [2, 4]; no different from reassignment: xs = [2, 4, 6, 8]
xs[1:3] = [4.5, 5.5, 6.5] # quite different from x[2] = [1]
xs


[2, 4.5, 5.5, 6.5, 8]

- `del` works with slices. It deletes list members.

In [4]:
xs = [2, 5, 33, 65]
del xs[1:3]
xs

[2, 65]

#### In-place Operations on Lists
It pays to understand what a _method_ is. A 'method' in a programming language with _objects_ is a way to send them _messages_. In this view, objects become _receivers_. A call to a method transfers control to the body (code) that implements a behavior. Because methods are implemented _inside_ objects, they may have some context about them. In this respect, they are different from _functions_ which are _global in scope_. In Python, since everything is an object, anything we do with it is by the way of invoking a method directly or indirectly. Like in many other languages, Python uses the `.` operator on an object reference to invoke or call a method. For example, 
```
xs = []
xs.append(1) # appends an element to the (empty) list; allocates space appropriately.
```
- Operators `+` and `*` do what we expect. No surprises. Use augmented assignments. Remember, this is the Pythonic way of overloading operators.
- In-place operations or methods _mutate_ an object.

In [5]:
l1 = [1, 3]
l2 = [2, 4]
l1 + l2


[1, 3, 2, 4]

In [6]:
l1 = [1] 
l1 *= 10 # 10 one's: adds _9_ copies of [1] at the end of [1]
print(f"list: {l1} of length {len(l1)}")

list: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] of length 10


##### Non Mutating List Methods
- `count(e)`
- `index(e)`

##### Mutating List Methods
In these examples, `xs = [] # some list`
- `xs.append(x)`
- `xs.clear()`
- `xs.extend(ys)`, where `ys` is another `iterable`. This is equivalent to `xs += ys`.
- `xs.insert(i, x)`: Inserts `i` in `xs` at index `i`. Idiosyncratic. **Never** raises any exception, even when the index is invalid. For an empty list `ys` how is `ys[0] = 4` is not pardonable, but `ys.insert(0, 4)` is?

In [7]:
ys = [2]
ys.insert(10, 4) # no hey problema, but inserts 4 at index 1, automatically extending the list!
print(ys)
zs = []
zs[0] = 100 # BOOM! IndexError!

[2, 4]


IndexError: list assignment index out of range

- `pop(i=-1)`: Pops (returns and removes) the element at the given index off the list by default; shifts affected elements appropriately. Can raise an exception.

In [8]:
xs = [1, 3, 55, 66]
xs.pop()
print(xs) # [1, 3, 55]
xs.pop(-2) # pop the second-last element, i.e. 1
print(xs) # [1, 55]
xs.pop(2) # BOOM! IndexError, there is no element at that index.
print(xs)

[1, 3, 55]
[1, 55]


IndexError: pop index out of range

- `xs.remove(x)`: Removes the first occurrence of `x` from `xs`. Can raise an exception!

In [9]:
xs = [1, 1, 2]
xs.remove(1) # [1, 2]
print(xs)
xs.remove(1) # [2]
print(xs)
xs.remove(1) # BOOM! ValueError, there is no such value in xs. Could it have silently ignored this as not 'exceptional'?

[1, 2]
[2]


ValueError: list.remove(x): x not in list

- `xs.reverse()`: Reverses a list in place.
- `xs.sort(key=None, reverse=False)`: Sorts the given list of _comparable_ objects. 

In [10]:
xs = [2, 53, 24, 66, 100]
xs.sort()
print(xs)


[2, 24, 53, 66, 100]


- In general, however, sorting is tricky! It is a crucial operation and the idea of comparing items of a (totally ordered) collection is at its heart. Java handles it by the means of, you guessed it right, an _interface_--`Comparable` (or, more generally, the `Comparator` interface); an iterable could be sorted only if its members are _comparable_ to each other. Since Java makes sorting harder for heterogeneous containers, it's not an issue there. I need to investigate more how heterogeneous collections (or containers) are handled by Python. 

In [11]:
xs = [1, [2, 3]]
xs.sort(reverse=True)


TypeError: '<' not supported between instances of 'int' and 'list'

- As the above snippet shows, it raises an error at runtime. This is expected. How can we _compare_ an `int` with an `str`, even if they happen to be members of a collection? A heterogeneous container simply can't have an ordering imposed on it. Period.

- Python's syntax is clearly more pleasant (less verbose) than Java's. Java's `Comparator`s and `Comparable`s are replaced by a ... right, `function` in Python. The _keyword argument_ (we'll learn about this shortly) named _key_, if provided by the caller of `sort`, must be a (reference to a) function which is apparently invoked with the members of the collection (`xs`) being compared. It's hoped that the values returned by the `key` function are comparable (that is, they are of built-in types that are comparable by definition) and sorting proceeds according to the criteria imposed by that function.
- Consider that we want to sort strings by length, rather than by lexical order. Thus, `"back"` precedes `"ahead"`, but follows "`cat`" and ties are broken arbitrarily. How might we proceed? Easy. Provide `len` which refers to a global function as the _value_ as the actual argument of the formal parameter named `key`:

In [12]:
words = ["ahead", "cat", "back", "in", "at"]
words.sort(key = len)
print(words) # note that the original order is preserved in case of ties, "in" precedes "at".

['in', 'at', 'cat', 'back', 'ahead']


- Suppose you wanted to first sort by length first, and then by lexical order (e.g. `at` precedes `in`), then? Of the top of my head, I can't think of how that might be made to work, because the `key` function only takes one parameter, not two. How does Python implement that? Should I resort to a web search or ChatGPT? ...
- I had to do a web search! It reveals that I have to return a tuple that simply contains _a pair of comparable values of the list members_ that Python compares for me. **This is quite unlike Java which just cedes the control of comparing two values of the collection entirely to your implementation of the _comparator interface_**:

In [13]:
def len_lexicon(s): # only for strings!
    return (len(s), s.lower()) # shorter strings first, then lexically preceding first
words = ["ahead", "cat", "back", "in", "at"]
words.sort(key = len_lexicon)
print(words) # now "at" precedes "in"

['at', 'in', 'cat', 'back', 'ahead']


- If I have more criteria for breaking the ties, well, I guess I need to add a third element to the returned tuple. Finding a third comparison criterion for strings is challenging because length (`int`) and lexical order (also `int` because Unicode is, after all, a number) break most ties and ensure a stable ordering. Lexical order takes care of _case_ as well because 'P' precedes 'p'. Thus, "in" and "IN" have the same length and lexical rank. Is there anything else based on which we can _order_ these two?
- It may be more useful with an object (class) that models the full name (first, last, middle initial) of a person. Instances can then be first compared on last name, then first name, and the middle initial. This seems straightforward, although I will refrain from adding it here because we haven't looked at OO Python yet. Conceptually, however, this is straightforward: Python wants your _comprator_ function (the actual argument for the formal parameter `key`) to return an n-tuple of comparable values of built-in types that it will then use for comparison. 

- The other keyword argument is `reverse` which may be thought as a convenience (perhaps because "reverse-sorting" is ubiquitous), but it appears to me that, given Python's philosophy, it is a must. In languages like Java, you implement your comparator comparing two values (`a, b`) and returning one of `-1, 0, 1` respectively if `a < b, a == b, a > b`, specifying the order of their comparison. But, since in Python the `key` function only receives a single argument, it needs some other way to configure the _direction_ of sorting i.e. ascending (default: `reverse = False`) or descending (`reverse = True`); the `key` function alone cannot achieve that, in general. Some _inversion tricks_ can be used (e.g. relying on properties of whole numbers: `if a < b, then -a > -b`) but doing that for strings or other objects would be harder. Python seems to have taken an easy way out (I haven't checked the literature, perhaps there's some discussion about this on a Python mailing list).

- There is a rather misleadingly named global function `sorted` which is analogous to `list.sort` method. `sorted` takes any `iterable` as its first argument and returns a sorted `iterable` based on `key` and `reverse` argument values. I haven't checked the code, but finally perhaps the same code is called at least in the CPython implementation? There is of course an important difference between `list.sort` and `sorted` in that **the former sorts in place and latter doesn't**.

### Set Operations
- `<class 'set'>` is the Python implementation of a mathematical set. Sets are `iterable`, but, unlike sequences, have no defined order of iteration. However, they are not inherently ordered, unlike, for example, the set of natural numbers, $\mathbb{N}$, or the set of real numbers, $\mathbb{R}$.

#### Set Membership
- Use the `in` or its inverse operator, `not in`, to test membership: 

In [8]:
S = {-2, 44}
print(2 in S)
print(44 in S)
print(2 not in S)
print(44 not in S)
print((2 not in S) == (not (2 in S)))

False
True
True
False
True


# Reminder: Don't Get Hung up on Little Things. There's a Mountain to Climb Alone (Albeit with the Tacit Help of Many Kindhearted People and Memories is Available). Acknowledge Oddities, Get Stuck and Unstuck, Move on! Progress!

#### Non Mutating Set Methods
These provide the support to realize the mathematical set operations. I am not sure why these methods violate the naming conventions. For instance, I'd have preferred `is_subset` or `is_disjoint` as method names. And now we are stuck with them because of backward compatibility requirements.

- `S.copy()`: is like `set(S)`: constructs a new set from the given set or iterable. It keeps the contract of the set: A set does not allow duplicates. Clearly, the concept of "object equality" is central to the general contract of a set.
- `S1.difference(S2)`: Implements $S_1-S_2$; this is non-commutative. Can be written `S1-S2`.
- `S1.intersection(S2)`: Implements $S_1\cap S_2$. Can be written `S1 & S2`.
- `S1.isdisjoint(S2)`: `True` if $S_1\cap S_2=\varnothing$, `False` otherwise.
- `S1.issubset(S2)`: `True` if $S_1\subseteq S_2$, `False` otherwise. Can be written `S1 <= S2`.
- `S1.issuperset(S2)`: `True` if $S_1\supseteq S_2$, `False` otherwise. Can be written `S1 >= S2`.
- `S1.symmetric_differece`: Returns $(S_1\cup S_2)-(S_1\cap S_2)$. Can be written `S1 ^ S2`. It is also $(S_1-S_2)\cup(S_2-S_1)$.
- `S1.union(S2)`: Returns $S_1\cup S_2$. Can be written `S1 | S2`.

#### Mutating Set Methods
- `S.add(x)`. Note: Returns `None`. I'd have preferred for it to return a boolean indicating if the set changed as a result.

In [16]:
S = {1}
print(S.add(2))  # returns None
S.add(1)
print(S)

None
{1, 2}


- `S.clear()`. The meaning is clear.
- `S.remove(x)` and `S.discard(x)` are identical except that the former raises `KeyError` when `x` is not in the set, whereas the latter does it silently (and always returns `None`).

In [17]:
S = set()
print(S.discard(2)) # discards 2, and returns None (not True)
print(S)
print(S.discard(2)) # silently does nothing, returns None (not True or False)
S.remove(2) # raises KeyError


None
set()
None


KeyError: 2

- `S.pop()`: Removes and returns a random member of the set if any is available.

##### Caution: A Typical Pattern
Iterating over a collection is a recurring programming task: we get a container object, use some construct available in the programming language to visit its every member, filter it out if unwanted, process it otherwise. 

However, what happens if such a routine task _structurally_ modifies the collection itself? 

An important task of this sort is the _removal of a member from the container_ (of course, while still iterating over it). This must be done _safely_, otherwise the iteration order may be adversely affected. 

Why does such a routine operation (e.g. removing odd integers from an array of integers) bear cautionary words? After all, we have removed elements from indexed (and hence random access) structures like arrays or lists since the earliest days of our imperative programming career using the `for` loops. On careful analysis of the code that performs such deletion, however, we'll realize that we undertook the _bookkeeping_ task of maintaining the order by moving members around. In linked structures, it was easier because we _carefully reassigned the next and/or previous pointers without having to "move" anything around_. In _hashed_ structures, it was even more complicated because resizing may result in rehashing.

The higher-level "Removing while iterating facility" alleviates us from bookkeeping. It gives us an illusion that it will be taken care of. However, it requires that we follow certain contracts.

In Java, we have an _explicit_ iterator **through** which we remove a member if we must.  If we do something like

1. Create a list, `List<Iterator> xs = ...`. Assume it has enough elements to begin with.
2. Create an `Iterator` instance: `Iterator<Integer> it = xs.iterator();`
3. `it.remove()`
4. `xs.remove(0)`
5. `it.remove()` => Results in a `ConcurrentModificationException`: `it` detects a condition that the underlying data structure was structurally modified concurrently without letting it know (by using the direct call `xs.remove` on line 4).

This is a good thing. The programming language is helping you write _safe_ code.

In Python, it is more subtle. It provides two looping constructs:
1. The (potentially) destructive loop (also called a _draining loop_) using `while`
2. The non destructive loop using `for`.
3. A sequenced structure like a `list` allows structural modification while iterating, but a hashed structure like `set` disallows it. This may come as a surprise initially, but as we graduate our programming career, it makes sense.

In [1]:
S = {10, 30}
while S: # peculiar: there is no 'in'
    item = S.pop()
    # consume item

len(S) # 0, it's cleared, giving you a chance to examine each member by consuming it

0

In [5]:
S = {10, 30, 233}
for item in S: # note the 'in'
    # process item (we have a reference to it)
    if item % 2 == 1:
        S.remove(item) # however, this isn't allowed because it is a 'concurrent' modification


RuntimeError: Set changed size during iteration

In [1]:
xs = [53, 5, 22]
for x in xs:
    if x % 2 == 1:
        xs.remove(x)
print(xs) # note: 5 was skipped over, the collection still has odd numbers!

[5, 22]


The best way to implement the "may delete items while iterating over their collection" task is to use comprehensions, which are a beautiful syntactical construct. We will soon learn comprehensions, but here is a teaser:

In [9]:
xs = [53, 5, 22, 522, 26, 33, 7]
evens = [x for x in xs if (x % 2 == 0)] # or [x for x in xs if not (x % 2 == 1)]
print(evens)
del xs # or you could reassign: xs = [x for x in xs if (x % 2 == 0)]

[22, 522, 26]


Yes, this is not in-place, but it is Pythonic. Garbage collector releases the memory associated with objects that are guaranteed to be unreachable by any part of a running program.